<a href="https://colab.research.google.com/github/lack-of-sleep/2609Study/blob/main/15_16.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Section 15**
Work2Vec, 유사한 단어 정렬

gensim을 이용해 bin.gz 파일에서 모델 로드후 King-Man+Woman을 계산해서 얻은 벡터와 가장 가까운 단어를 거리순으로 정렬

In [6]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!ls -l /content/drive/MyDrive/GoogleNews-vectors-negative300.bin.gz

-rw------- 1 root root 1647046227 Sep 21 04:37 /content/drive/MyDrive/GoogleNews-vectors-negative300.bin.gz


In [7]:
!pip install gensim

import gensim

model = gensim.models.KeyedVectors.load_word2vec_format('/content/drive/MyDrive/GoogleNews-vectors-negative300.bin.gz', binary=True)

In [9]:
king_vec = model['king']
man_vec = model['man']
woman_vec = model['woman']
queen_vec = king_vec - man_vec + woman_vec
#벡터 계산

similar_words = model.most_similar(positive=[queen_vec], topn=10)
#most_similar를 이용해 queen과 positive로 유사한 단어를 10개 저장

for word, similarity in similar_words:
  print(f'{word}: {similarity}')

king: 0.8449392318725586
queen: 0.7300517559051514
monarch: 0.645466148853302
princess: 0.6156251430511475
crown_prince: 0.5818676352500916
prince: 0.5777117609977722
kings: 0.5613663792610168
sultan: 0.5376775860786438
Queen_Consort: 0.5344247817993164
queens: 0.5289887189865112


In [10]:
similar_words = model.most_similar(positive=[king_vec, woman_vec],negative=[man_vec] ,topn=10)
#King, woman이랑 비슷하지만 man이랑은 다른 벡터 10개

for word, similarity in similar_words:
  print(f'{word}: {similarity}')

#queen과 결과가 똑같다

king: 0.8449392318725586
queen: 0.7300517559051514
monarch: 0.645466148853302
princess: 0.6156251430511475
crown_prince: 0.5818676352500916
prince: 0.5777117609977722
kings: 0.5613663792610168
sultan: 0.5376775860786438
Queen_Consort: 0.5344247817993164
queens: 0.5289887189865112


In [11]:
from itertools import combinations

fruits = ["pen", "pineapple", "banana"]

combinations_of_fruits = combinations(fruits, 2)
#과일 두개로 된 목록 생성

for combination in combinations_of_fruits:
  cos_similarity = model.similarity(combination[0], combination[1])
  print(combination[0], combination[1], cos_similarity)
#두 단어의 유사성 출력


pen pineapple 0.06774566
pen banana 0.05356275
pineapple banana 0.6587538


# **Section 16**
Sentence-Transformer 활용해 유사도 측정

In [12]:
!pip install sentence_transformers

In [14]:
import torch
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/bert-base-nli-mean-tokens")
#bert 기반 사전 학습 모델 불러오기

sentences = ["I went river fishing and stood on the bank for a while.",
             "I went to bank for withdrawing money.",
             "I played fishing near the ocean."]

Embeddings = model.encode(sentences)
#문장을 벡터화하여 Embedding에 저장

s1 = torch.tensor(Embeddings[0])
s2 = torch.tensor(Embeddings[1])
s3 = torch.tensor(Embeddings[2])
#tensor 형태로 변환

cos_similarity = torch.cosine_similarity(s1,s2,dim=0)
print(f"s1-s2:{cos_similarity.item()}")

cos_similarity = torch.cosine_similarity(s1,s3,dim=0)
print(f"s1-s3:{cos_similarity.item()}")
#코사인 유사도 계산, 출력

#1,3 문장의 유사도가 1,2 문장의 유사도의 두배
#같은 bank라는 단어가 들어가도 동음이의어를 알고 있기 때문에 유사도가 낮게 나옴

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.77k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

s1-s2:0.3672090768814087
s1-s3:0.7351641058921814


BERT는 빈칸 채우기 작업, 다음문장 예측작업을 사전 학습한 모델